GDF-Net Weight-Space Model Soup, Real-Time Distillation, and 8-View GeM Ensemble on DSB-2018
Dataset: Data Science Bowl 2018 (536 train / 67 validation / 67 test images at 256x256).
Test results (Table 2 Panel A): 0.8955 Dice for GDF-Net (Base), 0.8941 Dice for GDF-Net (Soup, SV No KD), 0.8992 Dice (0.8265 mIoU) for GDF-Net (Real-Time) at 30.0 FPS, and 0.8974 Dice (9.18 px HD95) for GDF-Net (Ensemble, 8V).


1. Environment setup and dataset preparation.


In [1]:
"""
Colab Environment Setup & Multi-Dataset Downloader
Sets up Kaggle credentials, installs dependencies, downloads CVC-ClinicDB, Kvasir-SEG, DSB-2018,
and ensures canonical checkpoints are downloaded.
"""

import os
import sys
import json
import glob
import zipfile
import subprocess

def run_cmd(cmd):
    print(f"[*] Running: {' '.join(cmd) if isinstance(cmd, list) else cmd}", flush=True)
    subprocess.run(cmd, shell=isinstance(cmd, str), check=True)

def setup_colab_env():
    print("=" * 80, flush=True)
    print("Setting up Google Colab Environment for GDFNet Multi-Dataset Retraining", flush=True)
    print("=" * 80, flush=True)

    # 1. Install Python packages
    print("[*] Installing dependencies (timm, albumentations, etc.)...", flush=True)
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.12", "albumentations==1.4.15", "opencv-python-headless", "scipy", "kaggle"])

    # 2. Configure Kaggle
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    kpath = os.path.join(kaggle_dir, "kaggle.json")
    with open(kpath, "w") as f:
        json.dump({"username": os.environ["KAGGLE_USERNAME"], "key": os.environ["KAGGLE_KEY"]}, f)
    os.chmod(kpath, 0o600)
    print("[+] Kaggle API key configured successfully.", flush=True)

    # 3. Create directories
    data_dir = "/content/data"
    ckpt_dir = "/content/checkpoints"
    results_dir = "/content/multidataset_results"
    os.makedirs(data_dir, exist_ok=True)
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)

    # 4. Check & Download Checkpoints
    cvc_ckpt = os.path.join(ckpt_dir, "GDFNet_CVC_ConvFormerS18_isic2018_best.pth")
    if not os.path.exists(cvc_ckpt):
        print("[*] Downloading Canonical Checkpoints from Kaggle...", flush=True)
        run_cmd(["kaggle", "kernels", "output", "namnguynnnn/gdfnetv4-canonical-s18-polyp-nuclei", "-p", ckpt_dir])
    print(f"[+] Canonical Checkpoints present in {ckpt_dir}:")
    for f in sorted(glob.glob(os.path.join(ckpt_dir, "*.pth"))):
        print(f"    - {os.path.basename(f)} ({os.path.getsize(f) / (1024*1024):.2f} MB)", flush=True)

    # 5. Download CVC-ClinicDB
    cvc_dir = os.path.join(data_dir, "cvc-clinicdb")
    os.makedirs(cvc_dir, exist_ok=True)
    if not any(f.endswith((".png", ".jpg")) for f in glob.glob(os.path.join(cvc_dir, "**/*.*"), recursive=True)):
        print("[*] Downloading CVC-ClinicDB...", flush=True)
        run_cmd(["kaggle", "datasets", "download", "-d", "namnguynnnn/cvc-clinicdb", "-p", cvc_dir, "--unzip"])
    cvc_files = glob.glob(os.path.join(cvc_dir, "**/*.*"), recursive=True)
    print(f"[+] CVC-ClinicDB ready: {len(cvc_files)} files in {cvc_dir}", flush=True)

    # 6. Download Kvasir-SEG
    kvasir_dir = os.path.join(data_dir, "kvasir-seg")
    os.makedirs(kvasir_dir, exist_ok=True)
    if not any(f.endswith((".png", ".jpg")) for f in glob.glob(os.path.join(kvasir_dir, "**/*.*"), recursive=True)):
        print("[*] Downloading Kvasir-SEG...", flush=True)
        run_cmd(["kaggle", "datasets", "download", "-d", "namnguynnnn/kvasir-seg", "-p", kvasir_dir, "--unzip"])
    kvasir_files = glob.glob(os.path.join(kvasir_dir, "**/*.*"), recursive=True)
    print(f"[+] Kvasir-SEG ready: {len(kvasir_files)} files in {kvasir_dir}", flush=True)

    # 7. Download DSB-2018
    dsb_dir = os.path.join(data_dir, "dsb2018")
    os.makedirs(dsb_dir, exist_ok=True)
    dsb_stage1_train = os.path.join(dsb_dir, "stage1_train")
    if not os.path.exists(dsb_stage1_train) or len(os.listdir(dsb_stage1_train)) < 500:
        print("[*] Downloading DSB-2018 stage1_train from tanumoynandy/nuclei...", flush=True)
        try:
            from kaggle.api.kaggle_api_extended import KaggleApi
            api = KaggleApi()
            api.authenticate()
            api.dataset_download_file("tanumoynandy/nuclei", "nuclei_datasets/nuclei_datasets/stage1_train.zip", path=dsb_dir)
        except Exception as e:
            print(f"[!] Warning KaggleApi download failed: {e}. Trying CLI...", flush=True)
            run_cmd(["kaggle", "datasets", "download", "-d", "tanumoynandy/nuclei", "-p", dsb_dir])

        zip_candidates = glob.glob(os.path.join(dsb_dir, "**", "*.zip"), recursive=True)
        for z in zip_candidates:
            if "stage1_train" in z:
                print(f"[*] Extracting {z} -> {dsb_stage1_train}...", flush=True)
                os.makedirs(dsb_stage1_train, exist_ok=True)
                with zipfile.ZipFile(z, "r") as zr:
                    zr.extractall(dsb_stage1_train)
                os.remove(z)
                break
            elif "nuclei" in z:
                print(f"[*] Extracting {z}...", flush=True)
                with zipfile.ZipFile(z, "r") as zr:
                    zr.extractall(dsb_dir)
                os.remove(z)
                sub_z = glob.glob(os.path.join(dsb_dir, "**", "stage1_train.zip"), recursive=True)
                if sub_z:
                    os.makedirs(dsb_stage1_train, exist_ok=True)
                    with zipfile.ZipFile(sub_z[0], "r") as zr:
                        zr.extractall(dsb_stage1_train)
                    os.remove(sub_z[0])
                break
    dsb_samples = len(os.listdir(dsb_stage1_train)) if os.path.exists(dsb_stage1_train) else 0
    print(f"[+] DSB-2018 ready: {dsb_samples} image folders in {dsb_stage1_train}", flush=True)

    print("\n[+] All datasets and checkpoints verified and ready for training!", flush=True)

if __name__ == "__main__":
    setup_colab_env()


Setting up Google Colab Environment for GDFNet Multi-Dataset Retraining
[*] Installing dependencies (timm, albumentations, etc.)...
[*] Running: /usr/bin/python3 -m pip install -q timm==1.0.12 albumentations==1.4.15 opencv-python-headless scipy kaggle
[+] Kaggle API key configured successfully.
[+] Canonical Checkpoints present in /content/checkpoints:
    - GDFNet_CVC_ConvFormerS18_isic2018_best.pth (112.84 MB)
    - GDFNet_DSB2018_ConvFormerS18_isic2018_best.pth (112.84 MB)
    - GDFNet_Kvasir_ConvFormerS18_isic2018_best.pth (112.84 MB)
[*] Running: kaggle datasets download -d namnguynnnn/cvc-clinicdb -p /content/data/cvc-clinicdb --unzip
[+] CVC-ClinicDB ready: 1225 files in /content/data/cvc-clinicdb
[+] Kvasir-SEG ready: 2001 files in /content/data/kvasir-seg
Dataset URL: https://www.kaggle.com/datasets/tanumoynandy/nuclei
[*] Extracting /content/data/dsb2018/stage1_train.zip -> /content/data/dsb2018/stage1_train...
[+] DSB-2018 ready: 670 image folders in /content/data/dsb2018/st

2. DSB-2018 Asymmetric Focal-Tversky training, weight-space model soup, and 35-epoch knowledge distillation.


In [2]:
"""
train_dsb2018.py
========================================================================================
Comprehensive In-Domain Retraining of Weight Soup, 8-View GeM Ensemble & Real-Time Knowledge Distillation on DSB-2018:
  - Dataset: DSB-2018 (670 images: 536 train / 67 val / 67 test) - Resolution: 256x256
  - Backbone: Canonical GDFNet ConvFormer-S18 (29,503,181 Parameters, 0 Attention heads)
  - Base Pretrained Checkpoint: /content/checkpoints/GDFNet_DSB2018_ConvFormerS18_isic2018_best.pth

Pipeline:
  1. Merge individual nuclei instance masks into single ground truth masks
  2. Evaluate Base In-Domain Model: Single-View (SV 0.50) & Single-View + Bi-ALT
  3. Train GDF-Net (Soup / Ensemble) In-Domain Asymmetric Focal-Tversky (25 Epochs) -> M_asym
  4. Construct True In-Domain Weight-Space Model Soup (0.70 M1 + 0.30 M_asym)
     -> Evaluate GDF-Net (Soup / Ensemble): 8-View GeM MS-TTA + Bi-ALT
  5. Train GDF-Net (Real-Time KD) Knowledge Distillation Student (35 Epochs, Multi-Scale Teacher, EMA, LLRD)
     -> Evaluate GDF-Net (Real-Time KD): Single-View + Bi-ALT (High-Throughput 31.3 FPS)
     -> Evaluate GDF-Net (Distilled 4V-TTA): 4-View TTA + Bi-ALT
     -> Evaluate GDF-Net (Distilled 8V-TTA): 8-View GeM MS-TTA + Bi-ALT
  6. Export Checkpoints & Results to /content/multidataset_results/
========================================================================================
"""

import os
import sys
import glob
import time
import copy
import random
from typing import Dict, List, Tuple, Any

import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader

# Add local path to import models
sys.path.insert(0, "/content")
if "__file__" in globals():
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from models import (
    CanonicalGDFNet,
    ModelEMA,
    AsymmetricFocalTverskyLoss,
    EnsembleDistillationLoss,
    build_soup_state_dict,
    FastDataset,
    extract_single_model_views,
    fuse_views,
    calibrate_bidirectional_alt_on_val,
    evaluate_predictions
)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def discover_dsb2018_splits():
    candidate_roots = [
        "/content/data/dsb2018/stage1_train",
        "/content/data/dsb2018",
        "/content/dsb2018_work/stage1_train",
        "/kaggle/input/data-science-bowl-2018/stage1_train",
        "/kaggle/temp/stage1_train",
    ]
    orig_dir = None
    for c in candidate_roots:
        if os.path.exists(c) and len(os.listdir(c)) >= 500:
            orig_dir = c
            break

    if not orig_dir:
        raise FileNotFoundError("Could not find DSB-2018 stage1_train directory.")

    merged_mask_dir = "/content/data/dsb2018/fused_masks"
    os.makedirs(merged_mask_dir, exist_ok=True)

    sample_folders = sorted(os.listdir(orig_dir))
    all_imgs, all_msks = [], []

    for folder in sample_folders:
        folder_path = os.path.join(orig_dir, folder)
        if not os.path.isdir(folder_path):
            continue

        im_candidates = glob.glob(os.path.join(folder_path, "images", "*.png"))
        if not im_candidates:
            continue
        im_file = im_candidates[0]

        # Check or create fused mask
        fused_mk_path = os.path.join(merged_mask_dir, f"{folder}.png")
        if not os.path.exists(fused_mk_path):
            mask_files = glob.glob(os.path.join(folder_path, "masks", "*.png"))
            if not mask_files:
                continue
            first_m = cv2.imread(mask_files[0], cv2.IMREAD_GRAYSCALE)
            merged = np.zeros_like(first_m, dtype=np.uint8)
            for mf in mask_files:
                m_arr = cv2.imread(mf, cv2.IMREAD_GRAYSCALE)
                if m_arr is not None:
                    merged = np.maximum(merged, m_arr)
            cv2.imwrite(fused_mk_path, merged)

        all_imgs.append(im_file)
        all_msks.append(fused_mk_path)

    min_len = len(all_imgs)
    # Deterministic seed=42 split (536 / 67 / 67)
    combined = list(zip(all_imgs, all_msks))
    random.seed(42)
    random.shuffle(combined)
    shuffled_imgs, shuffled_msks = zip(*combined)

    tr_x, tr_y = list(shuffled_imgs[:536]), list(shuffled_msks[:536])
    va_x, va_y = list(shuffled_imgs[536:536 + 67]), list(shuffled_msks[536:536 + 67])
    te_x, te_y = list(shuffled_imgs[536 + 67:]), list(shuffled_msks[536 + 67:])

    print(f"[+] DSB-2018: Total {min_len} | Train: {len(tr_x)} | Val: {len(va_x)} | Test: {len(te_x)}", flush=True)
    return (tr_x, tr_y), (va_x, va_y), (te_x, te_y)


def main():
    set_seed(42)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print("=" * 85, flush=True)
    print("GDFNet ConvFormer-S18 Retraining on DSB-2018 (Weight Soup, 8-View GeM Ensemble & Real-Time Knowledge Distillation)", flush=True)
    print(f"Device: {device} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}", flush=True)
    print("=" * 85, flush=True)

    out_dir = "/content/multidataset_results"
    os.makedirs(out_dir, exist_ok=True)

    # 1. Prepare Data Loaders (Resolution: 256x256, Batch size: 16)
    (tr_x, tr_y), (va_x, va_y), (te_x, te_y) = discover_dsb2018_splits()
    img_size = 256
    batch_size = 16

    train_ds = FastDataset(tr_x, tr_y, img_size=img_size, is_train=True)
    val_ds   = FastDataset(va_x, va_y, img_size=img_size, is_train=False)
    test_ds  = FastDataset(te_x, te_y, img_size=img_size, is_train=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    # 2. Load Base In-Domain Checkpoint
    ckpt_path = "/content/checkpoints/GDFNet_DSB2018_ConvFormerS18_isic2018_best.pth"
    if not os.path.exists(ckpt_path):
        ckpt_path = "GDF-Net/results/canonical_s18_polyp_nuclei/GDFNet_DSB2018_ConvFormerS18_isic2018_best.pth"

    print(f"\n[*] Loading Base In-Domain Model from: {ckpt_path}", flush=True)
    base_model = CanonicalGDFNet(backbone_name="convformer_s18.sail_in22k_ft_in1k_384", pretrained=False, drop_path_rate=0.0).to(device)
    base_sd = torch.load(ckpt_path, map_location="cpu")
    base_model.load_state_dict(base_sd, strict=True)
    print(f"[+] Loaded strictly ({sum(p.numel() for p in base_model.parameters()):,} parameters).", flush=True)

    # 3. Evaluate Base In-Domain Baseline
    print("\n" + "=" * 50, flush=True)
    print("Step 1: Evaluating Base In-Domain Model", flush=True)
    print("=" * 50, flush=True)
    v_base_val, v_up_val, v_dn_val, val_gts, _ = extract_single_model_views(base_model, val_loader, device)
    v_base_te, v_up_te, v_dn_te, te_gts, te_names = extract_single_model_views(base_model, test_loader, device)

    val_sv_p = fuse_views(v_base_val, v_up_val, v_dn_val, mode="sv")
    te_sv_p  = fuse_views(v_base_te, v_up_te, v_dn_te, mode="sv")

    # Base SV (0.50)
    m_base_sv, _ = evaluate_predictions(te_sv_p, te_gts, te_names, alt_params=None, threshold=0.50)
    print(f"[+] Base Model SV (0.50)       | Dice: {m_base_sv['dice']:.4f} | IoU: {m_base_sv['iou']:.4f} | BF2: {m_base_sv['bf2']:.4f} | HD95: {m_base_sv['hd95']:.2f}", flush=True)

    # Base SV + Bi-ALT
    alt_base = calibrate_bidirectional_alt_on_val(val_sv_p, val_gts)
    m_base_alt, _ = evaluate_predictions(te_sv_p, te_gts, te_names, alt_params=alt_base)
    print(f"[+] Base Model SV + Bi-ALT     | Dice: {m_base_alt['dice']:.4f} | IoU: {m_base_alt['iou']:.4f} | BF2: {m_base_alt['bf2']:.4f} | HD95: {m_base_alt['hd95']:.2f}", flush=True)

    # 4. Retrain GDF-Net (Soup / Ensemble): Asymmetric Focal-Tversky (25 Epochs)
    print("\n" + "=" * 50, flush=True)
    print("Step 2: Training In-Domain Asymmetric Focal-Tversky Model (25 Epochs)", flush=True)
    print("=" * 50, flush=True)
    asym_model = CanonicalGDFNet(backbone_name="convformer_s18.sail_in22k_ft_in1k_384", pretrained=False, drop_path_rate=0.15).to(device)
    asym_model.load_state_dict(base_sd, strict=True)

    criterion_asym = AsymmetricFocalTverskyLoss(alpha=0.30, beta=0.70, gamma=1.3333, tversky_weight=1.2)
    optimizer_asym = torch.optim.AdamW(asym_model.parameters(), lr=3.5e-5, weight_decay=1e-4)
    epochs_asym = 25
    scheduler_asym = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_asym, T_max=epochs_asym, eta_min=1e-6)
    scaler_asym = GradScaler('cuda')

    best_asym_val_dice = -1.0
    best_asym_sd = None

    for ep in range(1, epochs_asym + 1):
        asym_model.train()
        total_loss, n_batches = 0.0, 0
        t0 = time.time()

        for x, y, _ in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer_asym.zero_grad(set_to_none=True)

            with autocast('cuda'):
                outputs = asym_model(x, return_deep_supervision=True)
                loss = criterion_asym(outputs, y)

            scaler_asym.scale(loss).backward()
            scaler_asym.unscale_(optimizer_asym)
            torch.nn.utils.clip_grad_norm_(asym_model.parameters(), max_norm=1.0)
            scaler_asym.step(optimizer_asym)
            scaler_asym.update()

            total_loss += loss.item()
            n_batches += 1

        scheduler_asym.step()
        ep_time = time.time() - t0
        avg_loss = total_loss / max(n_batches, 1)

        # Validation at each epoch
        asym_model.eval()
        v_base_v, v_up_v, v_dn_v, _, _ = extract_single_model_views(asym_model, val_loader, device)
        v_val_sv = fuse_views(v_base_v, v_up_v, v_dn_v, mode="sv")
        alt_res = calibrate_bidirectional_alt_on_val(v_val_sv, val_gts)
        val_dice = alt_res["val_alt_dice"]

        is_best = val_dice > best_asym_val_dice
        if is_best:
            best_asym_val_dice = val_dice
            best_asym_sd = {k: v.cpu().clone() for k, v in asym_model.state_dict().items()}
            tag = " [NEW BEST]"
        else:
            tag = ""

        print(f"  Asym Epoch [{ep:02d}/{epochs_asym:02d}] ({ep_time:.1f}s) | Loss: {avg_loss:.4f} | Val Dice: {val_dice:.4f}{tag}", flush=True)

    asym_model.load_state_dict(best_asym_sd, strict=True)
    asym_ckpt_save = os.path.join(out_dir, "GDFNet_DSB2018_AsymTversky_best.pth")
    torch.save(best_asym_sd, asym_ckpt_save)
    print(f"[+] Saved Best AsymTversky checkpoint to {asym_ckpt_save}", flush=True)

    # 5. Build True In-Domain GDF-Net (Soup / Ensemble) Model Soup (0.70 Base + 0.30 AsymTversky)
    print("\n" + "=" * 50, flush=True)
    print("Step 3: Evaluating GDF-Net (Soup / Ensemble) True Weight-Space Model Soup", flush=True)
    print("=" * 50, flush=True)
    soup_sd = build_soup_state_dict([base_sd, best_asym_sd], (0.70, 0.30))
    mode5_soup_model = CanonicalGDFNet(backbone_name="convformer_s18.sail_in22k_ft_in1k_384", pretrained=False, drop_path_rate=0.0).to(device)
    mode5_soup_model.load_state_dict(soup_sd, strict=True)
    mode5_ckpt_save = os.path.join(out_dir, "GDFNet_DSB2018_Mode5_Soup_best.pth")
    torch.save(soup_sd, mode5_ckpt_save)

    # Evaluate GDF-Net (Soup / Ensemble)
    s_val_base, s_val_up, s_val_dn, _, _ = extract_single_model_views(mode5_soup_model, val_loader, device)
    s_te_base, s_te_up, s_te_dn, _, _ = extract_single_model_views(mode5_soup_model, test_loader, device)

    # Soup SV (0.50)
    s_te_sv = fuse_views(s_te_base, s_te_up, s_te_dn, mode="sv")
    m_mode5_sv, _ = evaluate_predictions(s_te_sv, te_gts, te_names, threshold=0.50)

    # Soup 8-View GeM MS-TTA + Bi-ALT (GDF-Net (Soup / Ensemble) Final)
    s_val_tta8 = fuse_views(s_val_base, s_val_up, s_val_dn, mode="tta8")
    s_te_tta8  = fuse_views(s_te_base, s_te_up, s_te_dn, mode="tta8")
    alt_mode5 = calibrate_bidirectional_alt_on_val(s_val_tta8, val_gts)
    m_mode5_final, _ = evaluate_predictions(s_te_tta8, te_gts, te_names, alt_params=alt_mode5)

    print(f"[+] GDF-Net (Soup, SV No KD)        | Dice: {m_mode5_sv['dice']:.4f} | IoU: {m_mode5_sv['iou']:.4f} | BF2: {m_mode5_sv['bf2']:.4f} | HD95: {m_mode5_sv['hd95']:.2f}", flush=True)
    print(f"[+] GDF-Net (Ensemble, 8V)       | Dice: {m_mode5_final['dice']:.4f} | IoU: {m_mode5_final['iou']:.4f} | BF2: {m_mode5_final['bf2']:.4f} | HD95: {m_mode5_final['hd95']:.2f}", flush=True)

    # 6. Retrain GDF-Net (Real-Time KD): Knowledge Distillation Student (35 Epochs)
    print("\n" + "=" * 50, flush=True)
    print("Step 4: Training GDF-Net (Real-Time KD) Knowledge Distillation Student (35 Epochs)", flush=True)
    print("=" * 50, flush=True)
    student = CanonicalGDFNet(backbone_name="convformer_s18.sail_in22k_ft_in1k_384", pretrained=False, drop_path_rate=0.15).to(device)
    student.load_state_dict(soup_sd, strict=True)
    student_ema = ModelEMA(student, decay=0.996)

    teacher_m1 = base_model.eval()
    teacher_asym = asym_model.eval()

    criterion_kd = EnsembleDistillationLoss(alpha_fp=0.42, beta_fn=0.58, gamma=1.25, pos_weight=1.18, kd_weight=0.55, temp=2.0)
    optimizer_kd = torch.optim.AdamW(student.parameters(), lr=4.0e-5, weight_decay=1e-4)
    epochs_kd = 35
    scheduler_kd = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_kd, T_max=epochs_kd, eta_min=1e-6)
    scaler_kd = GradScaler('cuda')

    best_kd_val_dice = -1.0
    best_student_sd = None

    for ep in range(1, epochs_kd + 1):
        student.train()
        total_loss, n_batches = 0.0, 0
        t0 = time.time()

        for x, y, _ in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer_kd.zero_grad(set_to_none=True)

            with torch.no_grad(), autocast('cuda'):
                # Teacher ensemble: 0.70 Base + 0.30 AsymTversky fused with GeM (p=1.5)
                z1 = teacher_m1(x)
                z2 = teacher_asym(x)
                p1_t = torch.clamp(torch.sigmoid(z1), 1e-6, 1.0 - 1e-6)
                p2_t = torch.clamp(torch.sigmoid(z2), 1e-6, 1.0 - 1e-6)
                gem_p = 1.5
                teacher_prob = torch.pow(0.70 * torch.pow(p1_t, gem_p) + 0.30 * torch.pow(p2_t, gem_p), 1.0 / gem_p)
                teacher_logits = 0.70 * z1 + 0.30 * z2

            with autocast('cuda'):
                outputs = student(x, return_deep_supervision=True)
                loss = criterion_kd(outputs, y, teacher_prob, teacher_logits)

            scaler_kd.scale(loss).backward()
            scaler_kd.unscale_(optimizer_kd)
            torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=1.0)
            scaler_kd.step(optimizer_kd)
            scaler_kd.update()

            student_ema.update(student)
            total_loss += loss.item()
            n_batches += 1

        scheduler_kd.step()
        ep_time = time.time() - t0
        avg_loss = total_loss / max(n_batches, 1)

        # Evaluate both raw and EMA student on validation Bi-ALT
        student.eval()
        v_base_r, v_up_r, v_dn_r, _, _ = extract_single_model_views(student, val_loader, device)
        v_val_r = fuse_views(v_base_r, v_up_r, v_dn_r, mode="sv")
        alt_r = calibrate_bidirectional_alt_on_val(v_val_r, val_gts)

        v_base_e, v_up_e, v_dn_e, _, _ = extract_single_model_views(student_ema.ema, val_loader, device)
        v_val_e = fuse_views(v_base_e, v_up_e, v_dn_e, mode="sv")
        alt_e = calibrate_bidirectional_alt_on_val(v_val_e, val_gts)

        cur_best = max(alt_r["val_alt_dice"], alt_e["val_alt_dice"])
        if cur_best > best_kd_val_dice:
            best_kd_val_dice = cur_best
            winning = student_ema.ema if alt_e["val_alt_dice"] >= alt_r["val_alt_dice"] else student
            best_student_sd = {k: v.cpu().clone() for k, v in winning.state_dict().items()}
            tag = " [NEW BEST KD]"
        else:
            tag = ""

        print(f"  KD Epoch [{ep:02d}/{epochs_kd:02d}] ({ep_time:.1f}s) | Loss: {avg_loss:.4f} | Raw: {alt_r['val_alt_dice']:.4f} | EMA: {alt_e['val_alt_dice']:.4f}{tag}", flush=True)

    mode6_ckpt_save = os.path.join(out_dir, "GDFNet_DSB-2018_Mode6_Distilled_best.pth")
    torch.save(best_student_sd, mode6_ckpt_save)
    print(f"[+] Saved Best Distilled Student checkpoint to {mode6_ckpt_save}", flush=True)

    student.load_state_dict(best_student_sd, strict=True)
    student.eval()

    # 7. Evaluate Distilled Model: SV, SV+Bi-ALT (GDF-Net (Real-Time KD)), 4-View TTA (GDF-Net (Distilled 4V-TTA)), 8-View GeM MS-TTA (GDF-Net (Distilled 8V-TTA))
    print("\n" + "=" * 50, flush=True)
    print("Step 5: Comprehensive Distilled Student Evaluation", flush=True)
    print("=" * 50, flush=True)

    st_val_base, st_val_up, st_val_dn, _, _ = extract_single_model_views(student, val_loader, device)
    st_te_base, st_te_up, st_te_dn, _, _ = extract_single_model_views(student, test_loader, device)

    # Distilled SV (0.50)
    st_te_sv = fuse_views(st_te_base, st_te_up, st_te_dn, mode="sv")
    m_kd_sv, _ = evaluate_predictions(st_te_sv, te_gts, te_names, threshold=0.50)

    # Distilled SV + Bi-ALT (GDF-Net (Real-Time KD) - High FPS target)
    st_val_sv = fuse_views(st_val_base, st_val_up, st_val_dn, mode="sv")
    alt_kd_sv = calibrate_bidirectional_alt_on_val(st_val_sv, val_gts)
    m_kd_sv_alt, _ = evaluate_predictions(st_te_sv, te_gts, te_names, alt_params=alt_kd_sv)

    # Distilled 4-View TTA + Bi-ALT (GDF-Net (Distilled 4V-TTA))
    st_val_tta4 = fuse_views(st_val_base, st_val_up, st_val_dn, mode="tta4")
    st_te_tta4  = fuse_views(st_te_base, st_te_up, st_te_dn, mode="tta4")
    alt_kd_tta4 = calibrate_bidirectional_alt_on_val(st_val_tta4, val_gts)
    m_kd_tta4, _ = evaluate_predictions(st_te_tta4, te_gts, te_names, alt_params=alt_kd_tta4)

    # Distilled 8-View GeM MS-TTA + Bi-ALT (GDF-Net (Distilled 8V-TTA))
    st_val_tta8 = fuse_views(st_val_base, st_val_up, st_val_dn, mode="tta8")
    st_te_tta8  = fuse_views(st_te_base, st_te_up, st_te_dn, mode="tta8")
    alt_kd_tta8 = calibrate_bidirectional_alt_on_val(st_val_tta8, val_gts)
    m_kd_tta8, _ = evaluate_predictions(st_te_tta8, te_gts, te_names, alt_params=alt_kd_tta8)

    # 8. Summary Table & Export
    records = [
        {"Dataset": "DSB-2018", "Mode": "GDF-Net (Base, tau=0.50)", "Dice": m_base_sv["dice"], "IoU": m_base_sv["iou"], "BF2": m_base_sv["bf2"], "HD95": m_base_sv["hd95"], "ASD": m_base_sv["asd"]},
        {"Dataset": "DSB-2018", "Mode": "GDF-Net (+Bi-ALT, SV)", "Dice": m_base_alt["dice"], "IoU": m_base_alt["iou"], "BF2": m_base_alt["bf2"], "HD95": m_base_alt["hd95"], "ASD": m_base_alt["asd"]},
        {"Dataset": "DSB-2018", "Mode": "GDF-Net (Soup, SV No KD)", "Dice": m_mode5_sv["dice"], "IoU": m_mode5_sv["iou"], "BF2": m_mode5_sv["bf2"], "HD95": m_mode5_sv["hd95"], "ASD": m_mode5_sv["asd"]},
        {"Dataset": "DSB-2018", "Mode": "GDF-Net (Ensemble, 8V MS-TTA)", "Dice": m_mode5_final["dice"], "IoU": m_mode5_final["iou"], "BF2": m_mode5_final["bf2"], "HD95": m_mode5_final["hd95"], "ASD": m_mode5_final["asd"]},
        {"Dataset": "DSB-2018", "Mode": "GDF-Net (Real-Time, SV KD, tau=0.50)", "Dice": m_kd_sv["dice"], "IoU": m_kd_sv["iou"], "BF2": m_kd_sv["bf2"], "HD95": m_kd_sv["hd95"], "ASD": m_kd_sv["asd"]},
        {"Dataset": "DSB-2018", "Mode": "GDF-Net (Real-Time, SV KD + Bi-ALT)", "Dice": m_kd_sv_alt["dice"], "IoU": m_kd_sv_alt["iou"], "BF2": m_kd_sv_alt["bf2"], "HD95": m_kd_sv_alt["hd95"], "ASD": m_kd_sv_alt["asd"]},
        {"Dataset": "DSB-2018", "Mode": "GDF-Net (Distilled, 4V-TTA + Bi-ALT)", "Dice": m_kd_tta4["dice"], "IoU": m_kd_tta4["iou"], "BF2": m_kd_tta4["bf2"], "HD95": m_kd_tta4["hd95"], "ASD": m_kd_tta4["asd"]},
        {"Dataset": "DSB-2018", "Mode": "GDF-Net (Distilled, 8V-TTA + Bi-ALT)", "Dice": m_kd_tta8["dice"], "IoU": m_kd_tta8["iou"], "BF2": m_kd_tta8["bf2"], "HD95": m_kd_tta8["hd95"], "ASD": m_kd_tta8["asd"]},
    ]

    df = pd.DataFrame(records)
    csv_path = os.path.join(out_dir, "results_dsb2018.csv")
    df.to_csv(csv_path, index=False)
    print("\n" + "=" * 90, flush=True)
    print("FINAL SUMMARY RESULTS FOR DSB-2018:", flush=True)
    print("=" * 90, flush=True)
    print(df.to_string(index=False), flush=True)
    print("=" * 90, flush=True)
    print(f"[+] Saved results to {csv_path}", flush=True)

if __name__ == "__main__":
    main()


GDFNet ConvFormer-S18 Retraining on DSB-2018 (Weight Soup, 8-View GeM Ensemble & Real-Time Knowledge Distillation)
Device: cuda:0 | GPU: NVIDIA L4
[+] DSB-2018: Total 670 | Train: 536 | Val: 67 | Test: 67

[*] Loading Base In-Domain Model from: /content/checkpoints/GDFNet_DSB2018_ConvFormerS18_isic2018_best.pth
[+] Loaded strictly (29,503,181 parameters).

Step 1: Evaluating Base In-Domain Model
[+] Base Model SV (0.50)       | Dice: 0.8955 | IoU: 0.8213 | BF2: 0.9198 | HD95: 9.32
[+] Base Model SV + Bi-ALT     | Dice: 0.8966 | IoU: 0.8224 | BF2: 0.9245 | HD95: 9.38

Step 2: Training In-Domain Asymmetric Focal-Tversky Model (25 Epochs)
  Asym Epoch [01/25] (11.4s) | Loss: 0.6419 | Val Dice: 0.8779 [NEW BEST]
  Asym Epoch [02/25] (6.7s) | Loss: 0.6314 | Val Dice: 0.8770
  Asym Epoch [03/25] (6.7s) | Loss: 0.6297 | Val Dice: 0.8762
  Asym Epoch [04/25] (6.7s) | Loss: 0.6303 | Val Dice: 0.8759
  Asym Epoch [05/25] (6.7s) | Loss: 0.6300 | Val Dice: 0.8778
  Asym Epoch [06/25] (6.7s) | Loss